# ❄️ Snowflake Data Loading — Detailed Notes with Examples

---

## 🧠 Overview

Snowflake supports two main data loading approaches:

1. **Bulk Loading (COPY INTO)**
2. **Continuous Loading (Snowpipe)**

---

# 📦 1. BULK LOADING (COPY INTO)

## 🔹 What is it?

Bulk loading is used to load large volumes of data in batches using the `COPY INTO` command.

---

## 🔹 Architecture Flow

Local Files → Internal Stage → COPY → Table
Cloud Storage → External Stage → COPY → Table

---

## 🔹 Key Features

- Requires **Virtual Warehouse**
- Supports **parallel loading**
- Best for **ETL pipelines & migrations**
- Supports **transaction control**

---

## 🧪 Example 1: Internal Stage

### Step 1: Create Table

```sql
CREATE TABLE sales (
    id INT,
    name STRING,
    amount NUMBER
);
```

### Step 2: Create File Format

```sql
CREATE FILE FORMAT my_csv_format
TYPE = 'CSV'
FIELD_DELIMITER = ','
SKIP_HEADER = 1;
```

### Step 3: Upload File

```sql
PUT file://sales.csv @%sales;
```

### Step 4: Load Data

```sql
COPY INTO sales
FROM @%sales
FILE_FORMAT = my_csv_format;
```

---

## 🧪 Example 2: External Stage (S3)

```sql
CREATE STAGE my_s3_stage
URL='s3://mybucket/data/'
CREDENTIALS=(AWS_KEY_ID='xxx' AWS_SECRET_KEY='xxx')
FILE_FORMAT = my_csv_format;

COPY INTO sales
FROM @my_s3_stage
PATTERN='.*sales.*.csv';
```

---

## ⚡ When to Use COPY

- Historical data loads
- Daily batch jobs
- Data migrations

---

# 🔄 2. CONTINUOUS LOADING (SNOWPIPE)

## 🔹 What is Snowpipe?

Snowpipe is a **serverless ingestion service** that automatically loads new data as soon as it arrives.

---

## 🔹 Architecture

Files → Stage → Snowpipe → Table

---

## 🔹 Key Features

- No warehouse required
- Near real-time ingestion
- Fully managed (serverless)

---

# 🧩 3. PIPE

## 🔹 What is PIPE?

A PIPE is a Snowflake object that contains a COPY statement used by Snowpipe.

---

## 🧪 Example

```sql
CREATE PIPE mypipe AS
COPY INTO sales
FROM @my_stage
FILE_FORMAT = (TYPE = 'CSV');
```

---

## ⚡ Behavior

- Automatically executes when triggered
- Tracks ingestion history
- Can be paused/resumed

---

# 🌐 4. SNOWPIPE REST API

## 🔹 Use Case

Used when you manually trigger ingestion.

---

## Example

```bash
POST /v1/data/pipes/mypipe/insertFiles
```

---

# ⚡ 5. AUTO-INGEST (BEST PRACTICE)

## 🔹 What is it?

Automatically triggers Snowpipe when files arrive in cloud storage.

---

## 🔹 Flow

S3 → Event Notification → Snowpipe → Table

---

## 🧪 Example

```sql
CREATE PIPE sales_pipe
AUTO_INGEST = TRUE
AS
COPY INTO sales
FROM @sales_stage
FILE_FORMAT = (TYPE = 'JSON');
```

---

## 🔹 Supported Integrations

- AWS S3 (Event Notification)
- Azure Event Grid
- GCP Pub/Sub

---

# 🔥 6. USE CASES

## COPY (Batch)

- Data warehouse loads
- ETL pipelines
- Scheduled jobs

## SNOWPIPE (Streaming)

- Real-time analytics
- IoT data ingestion
- Log processing
- Kafka / Firehose pipelines

---

# ⚡ 7. BEST PRACTICES

## File Size

- Snowpipe: 10MB – 100MB
- COPY: Larger files preferred

---

## Performance

- Use compressed formats (gzip, parquet)
- Use parallel loading
- Partition data

---

## Cost Optimization

- Use COPY for bulk (cheaper)
- Use Snowpipe only when needed

---

# 🚀 8. END-TO-END EXAMPLE

## Scenario: Real-time sales ingestion

### Step 1: Create Stage

```sql
CREATE STAGE sales_stage
URL='s3://sales-bucket/'
CREDENTIALS=(AWS_KEY_ID='xxx' AWS_SECRET_KEY='xxx');
```

### Step 2: Create Table

```sql
CREATE TABLE sales (
    id INT,
    name STRING,
    amount NUMBER
);
```

### Step 3: Create Snowpipe

```sql
CREATE PIPE sales_pipe
AUTO_INGEST = TRUE
AS
COPY INTO sales
FROM @sales_stage
FILE_FORMAT = (TYPE = 'JSON');
```

### Step 4: Enable S3 Notification

- Configure S3 event to trigger Snowpipe

---

## ✅ Result

- File uploaded → Snowpipe triggered → Data loaded automatically

---

# 🧠 FINAL SUMMARY

- COPY → Batch loading
- Snowpipe → Real-time loading
- PIPE → Defines ingestion logic
- AUTO-INGEST → Fully automated pipeline

---
